In [1]:
import pandas as pd
import geopandas as gpd
import json
import os
import numpy as np

In [2]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"
analysis_data_folder = 'Analysis_Data_insight_Wetlands_Inventory'

In [3]:
locations = gpd.read_file(f'{gcs_path}/locations_simplified.geojson')
locations_sahel = locations[locations['type'] == 'global']
locations_countries = locations[locations['type'] == 'admin_region']
locations_wpda = locations[locations['type'] == 'wdpa']
locations_basins = locations[locations['type'] == 'hydro_basin']
print(f'Total locations: {len(locations)}')
print(f'Total Sahel locations: {len(locations_sahel)}')
print(f'Total country locations: {len(locations_countries)}')
print(f'Total WDPA locations: {len(locations_wpda)}')
print(f'Total basin locations: {len(locations_basins)}')

Total locations: 2068
Total Sahel locations: 1
Total country locations: 26
Total WDPA locations: 2024
Total basin locations: 17


## Wetland type per country

In [4]:
file_path = f"{gcs_path}/{analysis_data_folder}/Country_%25%20wetland%20type_Ramsar%20classification.shp"
ramsar_wetlands = gpd.read_file(file_path)
ramsar_wetlands['name_en'] = ramsar_wetlands['name_en'].replace({'The Gambia': 'Gambia'})
ramsar_wetlands = ramsar_wetlands.merge(locations_countries[['name', 'code']], left_on='name_en', right_on='name', how='left')
ramsar_wetlands.drop(columns=['name_x', 'name_y', 'fid', 'osm_id', 'name_en', 'boundary', 'admin_leve',
       'admin_cent', 'admin_ce_1', 'admin_ce_2', 'label_node', 'label_no_1',
       'label_no_2', 'layer', 'path', 'HISTO_0', 'HISTO_1', 'HISTO_2',
       'HISTO_3', 'tot', 'geometry'],
        inplace=True)
ramsar_wetlands.rename(columns={'GID_0': 'code'}, inplace=True)
ramsar_wetlands.dropna(subset=['code'], inplace=True)
ramsar_wetlands['location_id'] = ramsar_wetlands['code'].apply(lambda x: 'adminregion_' + x.lower())
ramsar_wetlands.drop(columns=['code'], inplace=True)
ramsar_wetlands.columns = ramsar_wetlands.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '_')
ramsar_wetlands.head(3)



,coastal,manmade,inland,location_id
0,30.86,1.02,68.12,adminregion_gmb
2,0.04,18.92,81.04,adminregion_mrt
3,20.65,9.42,69.93,adminregion_sen


In [5]:
ramsar_wetlands_long = ramsar_wetlands.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
ramsar_wetlands_long.sort_values(['location_id', 'wetland_type'], inplace=True)
ramsar_wetlands_long.head(3)

,location_id,wetland_type,percentage
10,adminregion_ben,coastal,1.89
60,adminregion_ben,inland,87.01
35,adminregion_ben,manmade,11.10


In [6]:
ramsar_wetlands_long['wetland_type'].unique()

array(['coastal', 'inland', 'manmade'], dtype=object)

## Wetland types by wdpa

In [7]:
file_path = f"{gcs_path}/{analysis_data_folder}/WDPA_%25_wetland%20type_Ramsar.shp"
ramsar_wdpa = gpd.read_file(file_path)
ramsar_wdpa.drop(columns=['fid', 'v_idris', 'ramsarid', 'officialna', 'iso3', 'country_en',
       'area_off', 'OBJECTID', 'WDPAID', 'WDPA_PID', 'PA_DEF', 'NAME',
       'ORIG_NAME', 'DESIG', 'DESIG_ENG', 'DESIG_TYPE', 'IUCN_CAT', 'INT_CRIT',
       'MARINE', 'REP_M_AREA', 'GIS_M_AREA', 'REP_AREA', 'GIS_AREA', 'NO_TAKE',
       'NO_TK_AREA', 'STATUS', 'STATUS_YR', 'GOV_TYPE', 'OWN_TYPE',
       'MANG_AUTH', 'MANG_PLAN', 'VERIF', 'METADATAID', 'SUB_LOC',
       'PARENT_ISO', 'SUPP_INFO', 'CONS_OBJ', 'layer', 'path', 'PARENT_I_1',
       'Area_new', 'HISTO_0', 'HISTO_1', 'HISTO_2', 'HISTO_3', 'HISTO_TOT', 'geometry'],
        inplace=True)
ramsar_wdpa['location_id'] = ramsar_wdpa.index.to_series().apply(lambda x: f"wdpa_{x}")
ramsar_wdpa.columns = ramsar_wdpa.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '')
ramsar_wdpa.head()


,coastal,manmade,inland,location_id
0,7.511,0.822,91.667,wdpa_0
1,0.000,0.000,100.000,wdpa_1
2,0.000,0.034,99.966,wdpa_2
3,0.000,2.873,97.127,wdpa_3
4,0.000,0.139,99.861,wdpa_4


In [8]:
ramsar_wdpa_long = ramsar_wdpa.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
ramsar_wdpa_long.sort_values(['location_id', 'wetland_type'], inplace=True)
ramsar_wdpa_long.head(3)

,location_id,wetland_type,percentage
0,wdpa_0,coastal,7.511
4048,wdpa_0,inland,91.667
2024,wdpa_0,manmade,0.822


## Wetland types by hydrobasin

In [9]:
file_path = f"{gcs_path}/{analysis_data_folder}/Hydrobasins_%25%20wetland%20type_Ramsar.shp"
ramsar_hydrobasins = gpd.read_file(file_path)
ramsar_hydrobasins.drop(columns=['NEXT_DOWN', 'NEXT_SINK', 'MAIN_BAS', 'DIST_SINK',
       'DIST_MAIN', 'SUB_AREA', 'UP_AREA', 'PFAF_ID', 'ENDO', 'COAST', 'ORDER',
       'SORT', 'HISTO_0', 'HISTO_1', 'HISTO_2', 'HISTO_3', 'HISTO_TOT','geometry'],
        inplace=True)
ramsar_hydrobasins['HYBAS_ID'] = ramsar_hydrobasins['HYBAS_ID'].astype(str)

ramsar_hydrobasins = pd.merge(ramsar_hydrobasins, locations_basins[['id', 'code']], left_on='HYBAS_ID', right_on='code', how='left')
ramsar_hydrobasins.drop(columns=['code', 'HYBAS_ID'], inplace=True)
ramsar_hydrobasins.dropna(subset=['id'], inplace=True)
ramsar_hydrobasins.columns = ramsar_hydrobasins.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '')
ramsar_hydrobasins.rename(columns={'id': 'location_id'}, inplace=True)
ramsar_hydrobasins_long = ramsar_hydrobasins.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')

ramsar_hydrobasins_long.sort_values(['location_id', 'wetland_type'], inplace=True)
ramsar_hydrobasins_long.head(3)

,location_id,wetland_type,percentage
4,hydrobasin_congo_basin,coastal,0.000
38,hydrobasin_congo_basin,inland,99.679
21,hydrobasin_congo_basin,manmade,0.321


## Wetland types Sahel

In [10]:
file_path = f"{gcs_path}/{analysis_data_folder}/Sahel_%25%20wetland%20type_Ramsar%20classification.shp"
ramsar_sahel = gpd.read_file(file_path)
ramsar_sahel.drop(columns=['fid', 'Id', 'Wetlands %', 'Wetlands_1', 'Wetlands_2', 'Wetlands_3',
       'Tot wet a', 'geometry'], inplace=True)
ramsar_sahel['location_id'] = 'global_sahel'
ramsar_sahel.columns = ramsar_sahel.columns.str.lower().str.replace('%', '').str.strip().str.replace(' ', '')
ramsar_sahel_long = ramsar_sahel.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
ramsar_sahel_long.sort_values(['location_id', 'wetland_type'], inplace=True)
ramsar_sahel_long.head(3)

,location_id,wetland_type,percentage
1,global_sahel,coastal,0.73
2,global_sahel,inland,81.71
0,global_sahel,manmade,17.57


In [11]:
print(len(ramsar_wetlands_long['wetland_type'].unique()))
print(len(ramsar_wdpa_long['wetland_type'].unique()))
print(len(ramsar_hydrobasins_long['wetland_type'].unique()))
print(len(ramsar_sahel_long['wetland_type'].unique()))

3
3
3
3


## Combine all data

In [12]:
ramsar_combined = pd.concat([ramsar_wetlands_long,
                          ramsar_wdpa_long,
                          ramsar_hydrobasins_long,
                          ramsar_sahel_long], ignore_index=True)

In [13]:
print(len(ramsar_combined['wetland_type'].unique()))
ramsar_combined['wetland_type'].unique()

3


array(['coastal', 'inland', 'manmade'], dtype=object)

In [14]:
ramsar_combined['percentage'] = ramsar_combined['percentage'].replace(np.nan, 0)

In [15]:
color_dict = {"coastal": "#3BCC9B",
      "manmade": "#FFA500",
      "inland": "#1E90FF"}


In [16]:
indicator_data_list = []

for location in ramsar_combined['location_id'].unique():
    df_location = ramsar_combined[ramsar_combined['location_id'] == location]
    df_location.reset_index(drop=True, inplace=True)
    data_list = []
    for i in range(len(df_location)):
        data_list.append({
            "id":"wetland_ramsar_" + location + "_" + str(i),
            "label": df_location.loc[i, 'wetland_type'],
            "value": df_location.loc[i, 'percentage'],
            "type":"",
            "group": "",
            "color": color_dict.get(df_location.loc[i, 'wetland_type'], "#000000"),
            "format": "number",
            "unit": "%"
        })
    data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": "wetland-ramsar-" + location,
                 "location": location,
                 "indicator": "wetland-types-ramsar",
                 "data": json.loads(data_json),
                 "locale": {"en": {"labels":{
                     'coastal':'Coastal wetland',
                     'inland':'Inland wetland',
                     'manmade':'Man-made wetland'
                    }
                    }},
                }
    indicator_data_list.append(temp_dict)


In [17]:
print(json.dumps(indicator_data_list[0], indent=2))

{
  "id": "wetland-ramsar-adminregion_ben",
  "location": "adminregion_ben",
  "indicator": "wetland-types-ramsar",
  "data": [
    {
      "id": "wetland_ramsar_adminregion_ben_0",
      "label": "coastal",
      "value": 1.89,
      "type": "",
      "group": "",
      "color": "#3BCC9B",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_ramsar_adminregion_ben_1",
      "label": "inland",
      "value": 87.01,
      "type": "",
      "group": "",
      "color": "#1E90FF",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_ramsar_adminregion_ben_2",
      "label": "manmade",
      "value": 11.1,
      "type": "",
      "group": "",
      "color": "#FFA500",
      "format": "number",
      "unit": "%"
    }
  ],
  "locale": {
    "en": {
      "labels": {
        "coastal": "Coastal wetland",
        "inland": "Inland wetland",
        "manmade": "Man-made wetland"
      }
    }
  }
}


### Save to json

In [18]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "wetland-ramsar-"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('wetland-ramsar-')]
# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)